In [1]:
import os
import json
import time
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI
from dotenv import load_dotenv

# Load the API key from this project's .env file
env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Check your .env file and working folder."
    )

# Initialize the client
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

print("✅ Setup complete! OpenAI client initialized.")

✅ Setup complete! OpenAI client initialized.


In [2]:
def call_openai(prompt, temperature=0.7):
    """Send one prompt and return the model's response."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )

    return response.choices[0].message.content


def run_tests(prompt, num_runs=5, temperature=0.7):
    """Run a prompt several times and record responses or errors."""

    def run_once(run_number):
        try:
            response = call_openai(prompt, temperature=temperature)
            return {
                "run": run_number,
                "response": response,
                "error": None
            }
        except Exception as error:
            return {
                "run": run_number,
                "response": None,
                "error": str(error)
            }

    # Run up to two requests at the same time
    with ThreadPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(
            run_once, range(1, num_runs + 1)
        ))

    for result in results:
        print(f"\n--- Run {result['run']} ---")
        if result["error"] is not None:
            print("ERROR:", result["error"])
        else:
            print(result["response"])

    return results


print("✅ Helper functions loaded!")

✅ Helper functions loaded!


In [3]:
# Initial zero-shot prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# Test the prompt once
sentiment_result_v1 = call_openai(sentiment_prompt_v1)

print("Sentiment Analysis Result:")
print(sentiment_result_v1)

Sentiment Analysis Result:
This customer message can be classified as positive feedback or a positive customer review.


### Initial observation — Sentiment v1
The model correctly identified positive sentiment, but returned a full sentence instead of a single label. The prompt does not specify an output format.

In [4]:
# Initial zero-shot prompt for product description
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

# Test the prompt once
product_result_v1 = call_openai(product_prompt_v1)

print("Product Description Result:")
print(product_result_v1)

Product Description Result:
**Product Name: SwiftConnect Wireless Mouse**

**Price: $29.99**

Elevate your productivity with the SwiftConnect Wireless Mouse, the perfect blend of style, comfort, and cutting-edge technology. Designed for seamless navigation, this sleek and ergonomic mouse fits perfectly in your hand, ensuring hours of comfortable use whether you're working at your desk or on the go.

**Key Features:**

- **Wireless Freedom**: Say goodbye to tangled cords and enjoy the flexibility of wireless connectivity. The SwiftConnect uses advanced 2.4GHz technology for a stable connection, ensuring responsiveness and accuracy without any lag.

- **Long Battery Life**: Equipped with an energy-efficient design, this mouse offers up to six months of battery life on a single AA battery, so you can focus on your tasks without worrying about frequent replacements.

- **Precision Tracking**: With a high-precision optical sensor and adjustable DPI settings (800, 1200, 1600), the SwiftConne

### Initial observation — Product v1
The response includes the correct product type and price, but invents a product name and unsupported specifications, such as battery life, DPI settings, and connection technology. It is also long because the prompt does not specify a length or structure.

In [5]:
# Initial zero-shot prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Test the prompt once
extraction_result_v1 = call_openai(extraction_prompt_v1)

print("Data Extraction Result:")
print(extraction_result_v1)

Data Extraction Result:
- **Order Number:** 12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged


### Initial observation — Extraction v1
The response extracted the date, delivery speed, and packaging condition correctly. However, it labeled the item identifier as an order number. It used a Markdown list because the prompt did not specify an output format.

In [6]:
# Run the initial sentiment prompt five times
sentiment_results_v1_5 = run_tests(
    sentiment_prompt_v1,
    num_runs=5,
    temperature=0.7
)


--- Run 1 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 2 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 3 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 4 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.

--- Run 5 ---
The customer message can be classified as **Positive Feedback** or **Satisfaction**.


### Sentiment v1 — 5-run analysis
All five responses correctly identified positive sentiment (100%). However, none returned only one of the target labels: positive, negative, or neutral (0% format compliance). Responses varied in wording, capitalization, and Markdown formatting. The initial prompt did not specify the required labels or format.

In [7]:
product_results_v1_5 = run_tests(
    product_prompt_v1,
    num_runs=5,
    temperature=0.7
)


--- Run 1 ---
**Product Description: Wireless Precision Mouse**

Upgrade your workspace with our Wireless Precision Mouse, designed for both comfort and performance at an unbeatable price of just $29.99. Say goodbye to tangled cords and hello to seamless navigation!

**Key Features:**

- **Ergonomic Design:** Our mouse fits snugly in your hand, reducing strain during long hours of use. Perfect for both work and play, its sleek contours ensure a natural grip.

- **Advanced Wireless Technology:** Enjoy a reliable connection with our cutting-edge 2.4GHz wireless technology. Experience smooth, lag-free tracking from up to 33 feet away!

- **High Precision Tracking:** Equipped with a high-precision optical sensor, this mouse delivers exceptional accuracy on various surfaces, making it ideal for design work, gaming, or everyday tasks.

- **Long-lasting Battery Life:** Forget the hassle of frequent battery changes! Our mouse features an energy-efficient design that provides up to 12 months o

### Product v1 — 5-run analysis
All five responses included the correct product type and price (100%). However, all invented unsupported product specifications, so factual compliance was 0%. Examples included 2.4 GHz connectivity, battery life, adjustable DPI, and compatibility. Product names and details varied between runs. The responses generally followed a similar structure with an introduction, feature list, and call to action.

In [8]:
extraction_results_v1_5 = run_tests(
    extraction_prompt_v1,
    num_runs=5,
    temperature=0.7
)


--- Run 1 ---
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 2 ---
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 3 ---
- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 4 ---
Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 5 ---
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Issue**: Damaged packaging


### Extraction v1 — 5-run analysis
All five responses correctly extracted the date, fast delivery, and damaged packaging. However, all mislabeled the item identifier as an order number. Formatting varied through introductory text and field names. None returned valid JSON, although the initial prompt did not request it. Repeated outputs can be consistent while still being incorrect.

In [9]:
print("SENTIMENT — V1 — 10 RUNS")
sentiment_results_v1_10 = run_tests(
    sentiment_prompt_v1,
    num_runs=10,
    temperature=0.7
)

SENTIMENT — V1 — 10 RUNS

--- Run 1 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.

--- Run 2 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 3 ---
This customer message can be classified as positive feedback or a positive review.

--- Run 4 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 5 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 6 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 7 ---
This customer message can be classified as positive feedback or a positive review.

--- Run 8 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.

--- Run 9 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 10 ---
The cus

### Sentiment v1 — 10-run analysis
All ten responses correctly identified positive sentiment (100%). None returned only the target label positive (0% format compliance). Variations included alternative labels, capitalization, and Markdown formatting. The ten-run test confirmed the patterns observed in the five-run test without revealing new failure types.

In [10]:
print("PRODUCT — V1 — 10 RUNS")
product_results_v1_10 = run_tests(
    product_prompt_v1,
    num_runs=10,
    temperature=0.7
)

PRODUCT — V1 — 10 RUNS

--- Run 1 ---
**Product Description: Wireless Serenity Mouse**

Elevate your productivity with the Wireless Serenity Mouse, designed for comfort and efficiency at an unbeatable price of just $29.99. Say goodbye to tangled cords and hello to seamless navigation with a sleek, modern design that complements any workspace.

**Key Features:**

- **Ergonomic Design:** The Wireless Serenity Mouse fits perfectly in your hand, providing maximum comfort for hours of use, whether you're at your desk, in a coffee shop, or on the go.

- **Advanced Wireless Technology:** Enjoy a reliable connection with a 2.4GHz wireless signal that ensures smooth tracking and response times. Experience the freedom of movement without the hassle of wires.

- **Long-lasting Battery Life:** Equipped with a power-saving mode, this mouse can last up to six months on a single AA battery, making it the ideal companion for your busy lifestyle.

- **Precision Tracking:** With adjustable DPI settings 

### Product v1 — 10-run analysis
All ten responses included the correct product type and price (100%), but none avoided unsupported claims (0% factual compliance). The larger test revealed additional examples, including programmable buttons and a one-year warranty. Specifications also varied across runs, such as six versus twelve months of battery life and different DPI ranges. The general structure remained similar.

In [11]:
print("EXTRACTION — V1 — 10 RUNS")
extraction_results_v1_10 = run_tests(
    extraction_prompt_v1,
    num_runs=10,
    temperature=0.7
)

EXTRACTION — V1 — 10 RUNS

--- Run 1 ---
- **Item Number:** 12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 2 ---
- **Item Ordered**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 3 ---
Here is the extracted information from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery: Fast
- Packaging Condition: Damaged

--- Run 4 ---
- Order Number: #12345
- Order Date: March 15th
- Delivery: Fast
- Packaging Condition: Damaged

--- Run 5 ---
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 6 ---
Here’s the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery:** Fast
- **Packaging Condition:** Damaged

--- Run 7 ---
- **Order Number**: #1234

### Extraction v1 — 10-run analysis
All ten responses correctly extracted the date, fast delivery, and damaged packaging. Only runs 1 and 2 correctly identified 12345 as an item identifier (20%); the other eight labeled it as an order number. Field names and formatting also varied. None returned JSON, which was not requested in v1. Unlike the five-run test, this test showed both correct and incorrect interpretations of the identifier.

In [12]:
print("SENTIMENT — V1 — 15 RUNS")
sentiment_results_v1_15 = run_tests(
    sentiment_prompt_v1,
    num_runs=15,
    temperature=0.7
)

SENTIMENT — V1 — 15 RUNS

--- Run 1 ---
This customer message can be classified as "Positive Feedback" or "Satisfaction."

--- Run 2 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.

--- Run 3 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 4 ---
This customer message can be classified as **positive feedback** or **satisfaction**.

--- Run 5 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

--- Run 6 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 7 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.

--- Run 8 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 9 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Run 10 --

### Sentiment v1 — 15-run analysis
All fifteen responses correctly identified positive sentiment (100%). None returned only the target label positive (0% format compliance). Wording, capitalization, and Markdown formatting varied, confirming the patterns from the earlier tests. The initial prompt did not specify labels or output format. These results apply only to the positive message tested, not to sentiment classification in general.

In [13]:
print("PRODUCT — V1 — 15 RUNS")
product_results_v1_15 = run_tests(
    product_prompt_v1,
    num_runs=15,
    temperature=0.7
)

PRODUCT — V1 — 15 RUNS

--- Run 1 ---
**Product Description: Wireless Precision Mouse**

Elevate your computing experience with our Wireless Precision Mouse, designed for seamless connectivity and superior comfort. Priced at just $29.99, this sleek and stylish mouse is perfect for both home and office use, delivering performance and functionality without breaking the bank.

**Key Features:**

- **Wireless Freedom:** Enjoy the convenience of wireless technology with a reliable 2.4GHz connection. Say goodbye to tangled cords and enjoy a clutter-free workspace.
  
- **Ergonomic Design:** Crafted with comfort in mind, the Wireless Precision Mouse features a contoured shape that fits perfectly in your hand, reducing strain during extended use. Ideal for professionals, gamers, and students alike!

- **High Precision Tracking:** Equipped with advanced optical sensors, this mouse offers smooth and accurate tracking on various surfaces. Whether you’re working on spreadsheets or navigating throu

All fifteen responses were complete. Response 15 was initially copied only partially; this was not a generation failure.

In [14]:
print("EXTRACTION — V1 — 15 RUNS")
extraction_results_v1_15 = run_tests(
    extraction_prompt_v1,
    num_runs=15,
    temperature=0.7
)

EXTRACTION — V1 — 15 RUNS

--- Run 1 ---
- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 2 ---
- **Order Number:** 12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 3 ---
- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

--- Run 4 ---
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 5 ---
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

--- Run 6 ---
Here is the extracted information from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery: Fast
- Packaging Condition: Damaged

--- Run 7

### Extraction v1 — 15-run analysis
All fifteen responses correctly extracted the date, fast delivery, and damaged packaging. Runs 7, 9, and 14 identified the number as an item identifier (20%). The other twelve mislabeled it as an order number. Field names and introductory text varied. None returned valid JSON, which was not requested in v1.

### V1 failure analysis — 15 runs

| Task | Evaluation criterion | Passing responses | Main problem |
|---|---|---|---|
| Sentiment | Correct positive sentiment | 15/15 (100%) | Alternative labels and formatting |
| Sentiment | Only the target label `positive` | 0/15 (0%) | Output format not specified |
| Product | Correct product type and price | 15/15 (100%) | These facts were preserved |
| Product | No unsupported factual claims | 0/15 (0%) | Invented names, specifications, and guarantees |
| Extraction | Correct item identifier interpretation | 3/15 (20%) | Item number confused with order number |
| Extraction | Valid JSON output | 0/15 (0%) | Output format not specified |

These metrics measure correctness and compliance with target requirements, not identical wording. The same criteria will be used when comparing improved versions. Results apply to the inputs tested.

In [15]:
# V2: Clear labels and output constraints
sentiment_prompt_v2 = """
Classify the sentiment of the customer message.

Choose exactly one label: positive, negative, or neutral.
Return only the lowercase label.
Do not add explanations, punctuation, or Markdown.

Customer message: "I love this product! It's exactly what I needed."
"""

sentiment_results_v2_15 = run_tests(
    sentiment_prompt_v2,
    num_runs=15,
    temperature=0.7
)


--- Run 1 ---
positive

--- Run 2 ---
positive

--- Run 3 ---
positive

--- Run 4 ---
positive

--- Run 5 ---
positive

--- Run 6 ---
positive

--- Run 7 ---
positive

--- Run 8 ---
positive

--- Run 9 ---
positive

--- Run 10 ---
positive

--- Run 11 ---
positive

--- Run 12 ---
positive

--- Run 13 ---
positive

--- Run 14 ---
positive

--- Run 15 ---
positive


### Sentiment v2 — 15-run analysis
All fifteen responses returned exactly positive. Sentiment accuracy remained at 100%, while target format compliance increased from 0% in v1 to 100% in v2. Explicit labels and output constraints removed the formatting variations observed earlier. This result applies to the positive message tested.

In [16]:
# V2: Structure, length, style, and factual constraints
product_prompt_v2 = """
Write a product description using only these facts:
- Product: wireless mouse
- Price: $29.99

Requirements:
- Use a friendly, professional tone.
- Write exactly three lines with these labels:
  Product:
  Description:
  Price:
- After Product:, write Wireless Mouse.
- After Description:, write 20 to 35 words.
- After Price:, write $29.99.
- You may mention freedom from a mouse cable.
- Do not invent a brand, model name, specifications, accessories,
  compatibility, warranties, or performance claims.
- Do not use Markdown, bullet points, or extra sections.
"""

product_results_v2_15 = run_tests(
    product_prompt_v2,
    num_runs=15,
    temperature=0.7
)


--- Run 1 ---
Product: Wireless Mouse  
Description: Enjoy the ultimate freedom of movement with our wireless mouse, designed for seamless navigation and comfort. Say goodbye to tangled cables and hello to effortless control.  
Price: $29.99

--- Run 2 ---
Product: Wireless Mouse  
Description: Enjoy the freedom of movement with our wireless mouse, designed for comfort and ease of use. Perfect for work or play, it eliminates the hassle of tangled cables.  
Price: $29.99

--- Run 3 ---
Product: Wireless Mouse  
Description: Enjoy the freedom of movement with our wireless mouse, designed for seamless navigation without the hassle of a cable. Perfect for both work and play, it combines convenience and style.  
Price: $29.99

--- Run 4 ---
Product: Wireless Mouse  
Description: Experience the freedom of movement with our wireless mouse, designed for comfort and convenience. Enjoy a clutter-free workspace without the hassle of tangled cables.  
Price: $29.99  

--- Run 5 ---
Product: Wirel

### Product v2 — 15-run analysis
All fifteen responses followed the required three-line structure and preserved the product type and price. However, none fully avoided unsupported claims. Specific invented specifications disappeared, but descriptions still claimed comfort, smooth navigation, style, or efficiency. Run 13 also claimed ergonomic design and reliable connectivity. Structure improved, but factual compliance remained at 0%.

In [17]:
extraction_prompt_v2 = """
Extract information from the customer feedback below.

Return only a valid JSON object with exactly these keys:
- "item_id": item identifier as a string, without the # symbol
- "order_date": date exactly as written; do not add a year
- "delivery_speed": delivery speed as a lowercase string
- "packaging_condition": packaging condition as a lowercase string

Use null for missing information.
Do not confuse an item identifier with an order number.
Do not add explanations, Markdown, or extra keys.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

extraction_results_v2_15 = run_tests(
    extraction_prompt_v2,
    num_runs=15,
    temperature=0.7
)


--- Run 1 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 2 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 3 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 4 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 5 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 6 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 7 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 8 ---
{
  "item_id": "12345",
  "order_date": "March 

### Extraction v2 — 15-run analysis
All fifteen responses returned valid JSON with exactly the required keys and correct values. Item identifier accuracy increased from 20% in v1 to 100% in v2, while JSON compliance increased from 0% to 100%. Clear field definitions and explicit constraints resolved the observed failures for this input.

In [18]:
# V3: Few-shot examples and explicit output constraints
sentiment_prompt_v3 = """
Classify customer sentiment as positive, negative, or neutral.

Examples:
Message: "The service was excellent. I am very happy."
Label: positive

Message: "The product broke immediately. I am disappointed."
Label: negative

Message: "My package arrived on Tuesday."
Label: neutral

Return exactly one lowercase label.
Do not add explanations, punctuation, or Markdown.

Message: "I love this product! It's exactly what I needed."
Label:
"""

sentiment_results_v3_15 = run_tests(
    sentiment_prompt_v3,
    num_runs=15,
    temperature=0.7
)


--- Run 1 ---
positive

--- Run 2 ---
positive

--- Run 3 ---
positive

--- Run 4 ---
positive

--- Run 5 ---
positive

--- Run 6 ---
positive

--- Run 7 ---
positive

--- Run 8 ---
positive

--- Run 9 ---
positive

--- Run 10 ---
positive

--- Run 11 ---
positive

--- Run 12 ---
positive

--- Run 13 ---
positive

--- Run 14 ---
positive

--- Run 15 ---
positive


### Sentiment v3 — 15-run analysis
All fifteen responses returned exactly positive (100% accuracy and format compliance). Few-shot prompting maintained the result achieved by v2, without a measurable additional improvement on this input. Task variations are needed to evaluate broader performance.

In [19]:
# V3: Step-by-step extraction and verification
extraction_prompt_v3 = """
Extract customer feedback information using these steps:
1. Identify the item identifier, order date, delivery speed,
   and packaging condition.
2. Distinguish item identifiers from order numbers.
3. Check each value against the feedback. Do not infer missing facts.
4. Build the JSON object and verify its keys and values.

Return only the final valid JSON object, without explanations
or Markdown, using exactly these keys:
- "item_id": string without the # symbol
- "order_date": date exactly as written, without adding a year
- "delivery_speed": lowercase string
- "packaging_condition": lowercase string

Use null for missing information.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

extraction_results_v3_15 = run_tests(
    extraction_prompt_v3,
    num_runs=15,
    temperature=0.7
)


--- Run 1 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 2 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 3 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 4 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 5 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 6 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 7 ---
{
  "item_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}

--- Run 8 ---
{
  "item_id": "12345",
  "order_date": "March 

### Extraction v3 — 15-run analysis
All fifteen responses returned valid JSON with the required keys and correct values (100% compliance). Step-by-step extraction and verification instructions maintained v2 performance without an additional measurable improvement on this input. Only the final output was evaluated.

In [20]:
product_prompt_v3 = """
Write a product description using only the supplied facts.

Output requirements:
- Exactly three lines: Product:, Description:, Price:
- Description must contain 20 to 35 words.
- Use a friendly, professional tone.
- Copy the supplied product name and price exactly.
- No Markdown or extra sections.

Factual requirements:
- Do not invent a brand, specifications, accessories, or guarantees.
- Do not claim comfort, ergonomics, reliability, speed, precision,
  efficiency, stylish design, or improved productivity unless supplied.
- For wireless products, you may mention freedom from a cable.
- When details are missing, invite the customer to check specifications
  rather than inventing them.

Example 1
Facts: Product = Wireless Keyboard; Price = $39.99
Output:
Product: Wireless Keyboard
Description: Choose a wireless keyboard for freedom from a keyboard cable. Available for $39.99, it is an option to consider; check the specifications to decide whether it meets your needs.
Price: $39.99

Example 2
Facts: Product = USB Cable; Price = $9.99
Output:
Product: USB Cable
Description: Looking for a USB cable? This option costs $9.99. Check the connector type, cable length, and device requirements before purchasing to confirm it suits your intended use.
Price: $9.99

Now write the description for these facts:
Product = Wireless Mouse
Price = $29.99

Return only the three required lines.
"""

product_results_v3_15 = run_tests(
    product_prompt_v3,
    num_runs=15,
    temperature=0.7
)


--- Run 1 ---
Product: Wireless Mouse  
Description: Explore the convenience of a wireless mouse for freedom from a cable. Priced at $29.99, it's a great choice; check the specifications to ensure it fits your needs.  
Price: $29.99

--- Run 2 ---
Product: Wireless Mouse  
Description: Explore the convenience of a wireless mouse, priced at $29.99. Enjoy the freedom from a cable while checking the specifications to ensure it fits your needs.  
Price: $29.99

--- Run 3 ---
Product: Wireless Mouse  
Description: Discover the convenience of a wireless mouse for $29.99. Enjoy the freedom from cables while navigating your tasks; please check the specifications to ensure it meets your needs.  
Price: $29.99

--- Run 4 ---
Product: Wireless Mouse  
Description: Explore the convenience of a wireless mouse for freedom from a cable. Priced at $29.99, it's a great choice; please check the specifications to ensure it meets your needs.  
Price: $29.99

--- Run 5 ---
Product: Wireless Mouse  
Descri

### Product v3 — 15-run analysis
All fifteen responses met the required structure, product name, price, and description length of 20–35 words. None invented specifications or unsupported technical qualities (100% factual compliance, compared with 0% in v1 and v2). General promotional opinions were accepted. Wording varied, but all responses met the requirements. Few-shot examples and explicit factual restrictions helped together; this test does not isolate their individual effects.

In [21]:
# Apply the same v3 instructions to new customer messages
original_message = "I love this product! It's exactly what I needed."

sentiment_variations = [
    (
        "negative",
        "The product arrived broken. I am very disappointed."
    ),
    (
        "neutral",
        "My package arrived on Tuesday."
    )
]

sentiment_variation_results = {}

for expected_label, message in sentiment_variations:
    test_prompt = sentiment_prompt_v3.replace(
        original_message, message
    )

    response = call_openai(test_prompt, temperature=0.7)

    sentiment_variation_results[expected_label] = {
        "message": message,
        "response": response,
        "passed": response.strip() == expected_label
    }

    print(f"\nMessage: {message}")
    print(f"Expected: {expected_label}")
    print(f"Response: {response}")
    print(f"Passed: {response.strip() == expected_label}")


Message: The product arrived broken. I am very disappointed.
Expected: negative
Response: negative
Passed: True

Message: My package arrived on Tuesday.
Expected: neutral
Response: neutral
Passed: True


In [22]:
new_message = "I would like to know when my order will arrive."

test_prompt = sentiment_prompt_v3.replace(
    original_message, new_message
)

response = call_openai(test_prompt, temperature=0.7)

sentiment_variation_results["neutral_unseen"] = {
    "message": new_message,
    "response": response,
    "passed": response.strip() == "neutral"
}

print("Message:", new_message)
print("Expected: neutral")
print("Response:", response)
print("Passed:", response.strip() == "neutral")

Message: I would like to know when my order will arrive.
Expected: neutral
Response: neutral
Passed: True


### Sentiment task variations
V3 correctly classified a new negative message and a new neutral question. Both returned the required single lowercase label. The first neutral test repeated a few-shot example, so an additional unseen neutral message was tested successfully. Each variation was tested once.

In [23]:
# Replace only the final product facts, preserving the examples
product_variation_prompt = product_prompt_v3.replace(
    "Product = Wireless Mouse\nPrice = $29.99",
    "Product = Water Bottle\nPrice = $14.99"
)

product_variation_response = call_openai(
    product_variation_prompt,
    temperature=0.7
)

print("PRODUCT VARIATION — Water Bottle")
print(product_variation_response)

PRODUCT VARIATION — Water Bottle
Product: Water Bottle  
Description: Stay hydrated with this water bottle, priced at $14.99. It’s a practical choice for daily use; please check the specifications to ensure it meets your hydration needs.  
Price: $14.99


### Product task variation
V3 produced a description for a Water Bottle priced at $14.99. It followed the three-line structure and the 20–35-word limit, without inventing material, capacity, or thermal properties. This variation was tested once.

In [24]:
original_feedback = (
    "I ordered item #12345 on March 15th. "
    "The delivery was fast but the packaging was damaged."
)

new_feedback = (
    "My order #98765 arrived slowly. "
    "The packaging was intact."
)

extraction_variation_prompt = extraction_prompt_v3.replace(
    original_feedback, new_feedback
)

extraction_variation_response = call_openai(
    extraction_variation_prompt,
    temperature=0.7
)

expected_extraction = {
    "item_id": None,
    "order_date": None,
    "delivery_speed": "slow",
    "packaging_condition": "intact"
}

print("EXTRACTION VARIATION")
print(extraction_variation_response)

try:
    extracted_data = json.loads(extraction_variation_response)
    print("Passed:", extracted_data == expected_extraction)
except json.JSONDecodeError:
    print("Passed: False — Invalid JSON")

EXTRACTION VARIATION
{
  "item_id": "98765",
  "order_date": null,
  "delivery_speed": "slowly",
  "packaging_condition": "intact"
}
Passed: False


### Extraction task variation — failure
The model incorrectly assigned order number 98765 to item_id instead of returning null. The missing date and intact packaging were handled correctly. The evaluator also expected slow, but the prompt allowed any lowercase delivery-speed string, so slowly was not itself a prompt violation. The prompt needs clearer identifier rules and explicit normalization.

In [25]:
# Refine v3 based on the failed variation
extraction_prompt_v3_final = extraction_prompt_v3.replace(
    "- \"delivery_speed\": lowercase string",
    '- "delivery_speed": "fast", "slow", or null'
).replace(
    "Use null for missing information.",
    """
Use null for missing information.

Identifier rules:
- Extract item_id only when the text explicitly identifies an item
  or product number.
- Never use an order number as item_id.
- If only an order number is present, item_id must be null.

Delivery-speed normalization:
- "fast", "quick", or "quickly" -> "fast"
- "slow" or "slowly" -> "slow"
- If speed is missing or unclear -> null
"""
)

# Retest the same variation
test_prompt = extraction_prompt_v3_final.replace(
    original_feedback, new_feedback
)

extraction_variation_response_final = call_openai(
    test_prompt,
    temperature=0.7
)

print(extraction_variation_response_final)

try:
    extracted_data = json.loads(extraction_variation_response_final)
    print("Passed:", extracted_data == expected_extraction)
except json.JSONDecodeError:
    print("Passed: False — Invalid JSON")

{
  "item_id": null,
  "order_date": null,
  "delivery_speed": "slow",
  "packaging_condition": "intact"
}
Passed: True


### Extraction variation — refinement
After adding explicit rules separating item identifiers from order numbers and defining delivery-speed normalization, the variation returned the expected JSON. This retest passed once; repeated testing is needed to assess consistency.

In [26]:
print("FINAL EVALUATION — SENTIMENT")
sentiment_results_final_15 = run_tests(
    sentiment_prompt_v3,
    num_runs=15,
    temperature=0.7
)

print("\nFINAL EVALUATION — PRODUCT")
product_results_final_15 = run_tests(
    product_prompt_v3,
    num_runs=15,
    temperature=0.7
)

print("\nFINAL EVALUATION — EXTRACTION")
extraction_results_final_15 = run_tests(
    extraction_prompt_v3_final,
    num_runs=15,
    temperature=0.7
)

FINAL EVALUATION — SENTIMENT

--- Run 1 ---
positive

--- Run 2 ---
positive

--- Run 3 ---
positive

--- Run 4 ---
positive

--- Run 5 ---
positive

--- Run 6 ---
positive

--- Run 7 ---
positive

--- Run 8 ---
positive

--- Run 9 ---
positive

--- Run 10 ---
positive

--- Run 11 ---
positive

--- Run 12 ---
positive

--- Run 13 ---
positive

--- Run 14 ---
positive

--- Run 15 ---
positive

FINAL EVALUATION — PRODUCT

--- Run 1 ---
Product: Wireless Mouse  
Description: Discover the convenience of a wireless mouse for freedom from a cable. Priced at $29.99, it's a great choice; please check the specifications to ensure it fits your needs.  
Price: $29.99

--- Run 2 ---
Product: Wireless Mouse  
Description: Explore the convenience of a wireless mouse for freedom from a cable. Priced at $29.99, it's a great choice; check the specifications to ensure it fits your needs.  
Price: $29.99

--- Run 3 ---
Product: Wireless Mouse  
Description: Discover the convenience of a wireless mouse fo

## Final evaluation and comparison

All tests used gpt-4o-mini with temperature=0.7. Compliance measures whether responses meet the task requirements, not whether their wording is identical.

| Evaluation criterion | V1 — 15 runs | V2 — 15 runs | V3 — 15 runs | Final — 15 runs |
|---|---|---|---|---|
| Sentiment: correct sentiment and exact label | 0% | 100% | 100% | 100% |
| Product: no unsupported factual claims | 0% | 0% | 100% | 100% |
| Extraction: valid JSON, required keys, correct values | 0% | 100% | 100% | 100% |

### Interpretation
Sentiment accuracy was already 100% in v1, but its output format did not meet the target requirement. Clear labels and constraints resolved this in v2. Product v2 improved structure but still invented qualities; examples and stronger factual restrictions resolved this in v3. Extraction v2 corrected the original identifier error and introduced valid JSON. V3 maintained that result.

### Task variations
New negative and neutral sentiment messages passed single-run tests. A water bottle description also met the product requirements. An extraction variation initially confused an order number with an item identifier. Explicit identifier rules and delivery-speed normalization corrected it in a single retest.

### Techniques and limitations
Few-shot examples were used for sentiment and product descriptions. Step-by-step extraction and verification instructions were used for extraction. Sentiment and extraction v3 did not improve the original-case scores beyond v2. Product examples and stronger restrictions were added together, so their individual effects cannot be isolated.

Final prompts met the >80% target on the original cases tested. Variations were tested only once, and the repaired extraction variation was not an independent test case. These results do not establish production readiness. Further evaluation should use unseen, diverse inputs and repeated variation tests.